In [1]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 52.4 MB/s eta 0:00:00


In [ ]:
import shutil
import os

source_file = "/kaggle/input/fulltextnew/arxiv_ai_full_text.csv"

destination_file = "/kaggle/working/arxiv_ai_full_text.csv"

os.makedirs(os.path.dirname(destination_file), exist_ok=True)

shutil.copy(source_file, destination_file)

print(f"File copied successfully from {source_file} to {destination_file}")

File copied successfully from /kaggle/input/fulltextnew/arxiv_ai_full_text.csv to /kaggle/working/arxiv_ai_full_text.csv


In [ ]:
import pandas as pd
import requests
import os
import fitz  # PyMuPDF
from tqdm import tqdm
import time

# --- CONFIGURATION ---
INPUT_CSV = '/kaggle/input/arxivmd/arxiv_ai_metadata.csv'
OUTPUT_CSV = '/kaggle/working/arxiv_ai_full_text.csv'
# A temporary directory to download PDFs one by one
PDF_DIR = 'research_papers_temp' 
# How often to save progress to the disk (in number of papers)
BATCH_SIZE = 50 

# --- SCRIPT ---

# 1. SETUP: Prepare directories and dataframes
os.makedirs(PDF_DIR, exist_ok=True)

print(f"Loading metadata from {INPUT_CSV}...")
try:
    input_df = pd.read_csv(INPUT_CSV)
    # Ensure a unique 'id' column exists, which is crucial for resuming.
    if 'id' not in input_df.columns:
        print("Error: The input CSV must have a unique 'id' column.")
        exit()
except FileNotFoundError:
    print(f"Error: The input file '{INPUT_CSV}' was not found.")
    exit()

# 2. RESUME LOGIC: Check for existing progress
processed_ids = set()
try:
    if os.path.exists(OUTPUT_CSV):
        print(f"Output file found. Reading processed papers to resume...")
        # Read only the 'id' column from the existing output to save memory
        processed_df = pd.read_csv(OUTPUT_CSV, usecols=['id'])
        processed_ids = set(processed_df['id'])
        print(f"Found {len(processed_ids)} papers already processed. Resuming.")
    else:
        print("No previous progress found. Starting from scratch.")
except Exception as e:
    print(f"Could not read existing output file. Starting fresh. Error: {e}")


# Filter the input dataframe to only include papers that haven't been processed yet
unprocessed_df = input_df[~input_df['id'].isin(processed_ids)].copy()

if unprocessed_df.empty:
    print("All papers have already been processed. Nothing to do.")
    exit()

print(f"\nStarting to process {len(unprocessed_df)} new papers...")

# 3. BATCH PROCESSING LOOP
results_batch = [] # A temporary list to hold results before saving

for index, row in tqdm(unprocessed_df.iterrows(), total=unprocessed_df.shape[0]):
    pdf_url = row.get('pdf_url')
    paper_id = row['id']
    
    extracted_text = ""
    if pdf_url and pd.notna(pdf_url):
        pdf_filename = f"{paper_id.replace('/', '_')}.pdf" # Sanitize filename
        pdf_path = os.path.join(PDF_DIR, pdf_filename)
        
        try:
            # A. Download the PDF
            response = requests.get(pdf_url, timeout=45)
            response.raise_for_status()
            with open(pdf_path, 'wb') as f:
                f.write(response.content)

            # B. Extract text
            with fitz.open(pdf_path) as doc:
                extracted_text = "".join(page.get_text() for page in doc)
            
            # C. Delete the PDF to save space
            os.remove(pdf_path)
            
        except Exception as e:
            print(f"\n[ERROR] Failed to process paper {paper_id}: {e}")
            # If an error occurs, clean up the downloaded file if it exists
            if os.path.exists(pdf_path):
                os.remove(pdf_path)
    
    # Add the result to the current batch
    row_data = row.to_dict()
    row_data['full_text'] = extracted_text
    results_batch.append(row_data)

    # 4. SAVE PROGRESS: When the batch is full, save it to the CSV
    if len(results_batch) >= BATCH_SIZE:
        batch_df = pd.DataFrame(results_batch)
        # Append to the CSV. Write header only if the file is new.
        batch_df.to_csv(OUTPUT_CSV, mode='a', header=not os.path.exists(OUTPUT_CSV) or os.path.getsize(OUTPUT_CSV) == 0, index=False)
        results_batch = [] # Reset the batch

# 5. FINAL SAVE: Save any remaining papers in the last batch
if results_batch:
    batch_df = pd.DataFrame(results_batch)
    batch_df.to_csv(OUTPUT_CSV, mode='a', header=not os.path.exists(OUTPUT_CSV) or os.path.getsize(OUTPUT_CSV) == 0, index=False)

print("\n--- Processing Finished ---")
print(f"All data with full text has been saved to {OUTPUT_CSV}")